In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('session8_polynomial_regression_data.csv')
df.head()

In [ ]:
# TASK 1: Transform Mobile Phone features (RAM, Storage) to Degree 2

# Using synthetic sample data to show transformation structure
sample_phone_data = np.array([[6, 128], 
                              [8, 256], 
                              [12, 256]])

poly_2 = PolynomialFeatures(degree=2)
transformed_features = poly_2.fit_transform(sample_phone_data)

print("Original Features Matrix [RAM, Storage]:\n", sample_phone_data)
print("\nFeature names mapping:", poly_2.get_feature_names_out(['RAM', 'Storage']))
print("\nResulting Degree 2 Feature Matrix:\n", transformed_features)
print("\n" + "="*50 + "\n")

In [ ]:
# TASK 2: Zomato Reviews vs Ratings (Linear vs Polynomial Degree 3)

X_zomato = df[['Zomato_Reviews']].values
y_zomato = df['Zomato_Ratings'].values

# Fit Linear Model
lin_model = LinearRegression()
lin_model.fit(X_zomato, y_zomato)

# Fit Polynomial Model (Degree 3)
poly_3 = PolynomialFeatures(degree=3)
X_zomato_poly3 = poly_3.fit_transform(X_zomato)
poly_model = LinearRegression()
poly_model.fit(X_zomato_poly3, y_zomato)

# Generate plotting points evenly across the range of reviews
X_plot = np.linspace(X_zomato.min(), X_zomato.max(), 500).reshape(-1, 1)
X_plot_poly3 = poly_3.transform(X_plot)

plt.figure(figsize=(9, 5))
plt.scatter(X_zomato, y_zomato, color='gray', alpha=0.5, label='Actual Ratings')
plt.plot(X_plot, lin_model.predict(X_plot), color='blue', linewidth=2, label='Linear Model (Degree 1)')
plt.plot(X_plot, poly_model.predict(X_plot_poly3), color='red', linewidth=2.5, label='Polynomial Model (Degree 3)')
plt.title('Zomato Reviews vs Ratings: Linear vs Polynomial Fit')
plt.xlabel('Number of Reviews')
plt.ylabel('Restaurant Rating (1-5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TASK 3: Instagram Non-linear Error Evaluation (Degrees 1 to 5)

X_insta = df[['Instagram_Posts']].values
y_insta = df['Instagram_Followers'].values

X_train, X_val, y_train, y_val = train_test_split(X_insta, y_insta, test_size=0.3, random_state=42)

train_errors = []
val_errors = []
degrees = range(1, 6)

for d in degrees:
    p_feat = PolynomialFeatures(degree=d)
    X_tr_poly = p_feat.fit_transform(X_train)
    X_va_poly = p_feat.transform(X_val)
    
    m = LinearRegression()
    m.fit(X_tr_poly, y_train)
    
    train_errors.append(mean_squared_error(y_train, m.predict(X_tr_poly)))
    val_errors.append(mean_squared_error(y_val, m.predict(X_va_poly)))

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_errors, label='Training Error', marker='o', color='blue')
plt.plot(degrees, val_errors, label='Validation Error', marker='s', color='orange')
plt.title('Instagram Model: Training vs Validation Error by Polynomial Degree')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error (MSE)')
plt.xticks(degrees)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TASK 4: Flipkart Price Prediction with L2 Regularization (Ridge)

X_flip = df[['Flipkart_Original_Price']].values
y_flip = df['Flipkart_Discounted_Price'].values

X_f_train, X_f_test, y_f_train, y_f_test = train_test_split(X_flip, y_flip, test_size=0.3, random_state=42)

# Degree 3 transformation
poly_flip = PolynomialFeatures(degree=3)
X_f_train_poly = poly_flip.fit_transform(X_f_train)
X_f_test_poly = poly_flip.transform(X_f_test)

# 1. Without Regularization
unreg_model = LinearRegression()
unreg_model.fit(X_f_train_poly, y_f_train)
unreg_pred = unreg_model.predict(X_f_test_poly)
unreg_mse = mean_squared_error(y_f_test, unreg_pred)

# 2. With L2 Regularization (Ridge)
reg_model = Ridge(alpha=1e5) # High alpha helps scale down high-degree variance
reg_model.fit(X_f_train_poly, y_f_train)
reg_pred = reg_model.predict(X_f_test_poly)
reg_mse = mean_squared_error(y_f_test, reg_pred)

print(f"Standard Polynomial (Degree 3) Test MSE: {unreg_mse:.2f}")
print(f"Ridge Regularized (Degree 3) Test MSE   : {reg_mse:.2f}")
print("\nExplanation: Regularization prevents high-degree equations from chasing data noise.")
print("It keeps coefficients smaller, thereby decreasing test error variance on unseen validation datasets.")
print("\n" + "="*50 + "\n")

In [ ]:
# TASK 5: AI-Generated Simulation Code for Underfitting & Overfitting

# Generating custom noisy curve dataset
np.random.seed(5)
X_sim = np.sort(np.random.uniform(-3, 3, 30)).reshape(-1, 1)
y_sim = 0.5 * X_sim**2 + X_sim + np.random.normal(0, 1, size=(30, 1))

X_sim_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

plt.figure(figsize=(12, 4))
degrees_sim = [1, 2, 15] # 1=Underfit, 2=Balanced/Just Right, 15=Overfit
titles = ['Underfitting (Degree 1)', 'Perfect Fit (Degree 2)', 'Overfitting (Degree 15)']

for i, d in enumerate(degrees_sim):
    plt.subplot(1, 3, i+1)
    p_sim = PolynomialFeatures(degree=d)
    X_sim_poly = p_sim.fit_transform(X_sim)
    
    m_sim = LinearRegression()
    m_sim.fit(X_sim_poly, y_sim)
    
    plt.scatter(X_sim, y_sim, color='black', s=30, label='Data')
    plt.plot(X_sim_plot, m_sim.predict(p_sim.transform(X_sim_plot)), color='red', linewidth=2)
    plt.title(titles[i])
    plt.ylim(-3, 8)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()